In [1]:


import joblib

X_train = joblib.load(
    "X_train_strat.pkl"
)

X_test = joblib.load(
    "X_test_strat.pkl"
)

y_train = joblib.load(
    "y_train_strat.pkl"
)

y_test = joblib.load(
    "y_test_strat.pkl"
)

In [2]:


import numpy as np
import optuna

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    f_regression
)

from sklearn.svm import SVR

from sklearn.model_selection import cross_validate

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

In [3]:


preprocessor = Pipeline([

    (
        "imputer",
        SimpleImputer(strategy="median")
    ),

    (
        "variance_filter",
        VarianceThreshold(threshold=0.01)
    ),

    (
        "scaler",
        StandardScaler()
    ),

    (
        "feature_selection",
        SelectKBest(
            score_func=f_regression,
            k=1000
        )
    )

])

In [6]:
from sklearn.model_selection import cross_validate
import numpy as np

def objective(trial):

    model = SVR(

        C=trial.suggest_float(
            "C",
            1e-2,
            1e3,
            log=True
        ),

        epsilon=trial.suggest_float(
            "epsilon",
            1e-4,
            1.0,
            log=True
        ),

        gamma=trial.suggest_categorical(
            "gamma",
            ["scale", "auto"]
        ),

        kernel="rbf"
    )

    pipeline = Pipeline([

        (
            "preprocessing",
            preprocessor
        ),

        (
            "model",
            model
        )

    ])

    scores = cross_validate(

        pipeline,

        X_train,

        y_train,

        cv=5,

        scoring={

            "r2": "r2",

            "mae": "neg_mean_absolute_error"

        },

        n_jobs=-1

    )

    mean_r2 = np.mean(
        scores["test_r2"]
    )

    mean_mae = -np.mean(
        scores["test_mae"]
    )

    trial.set_user_attr(
        "mean_r2",
        mean_r2
    )

    trial.set_user_attr(
        "mean_mae",
        mean_mae
    )

    print(
        f"Trial {trial.number} | "
        f"Mean R2 = {mean_r2:.4f} | "
        f"Mean MAE = {mean_mae:.4f}"
    )

    return mean_r2

In [7]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=30,
    show_progress_bar=True
)

[I 2026-06-08 16:38:30,599] A new study created in memory with name: no-name-60600a89-bcfe-4095-ba57-52a87e318315


  0%|          | 0/30 [00:00<?, ?it/s]

Trial 0 | Mean R2 = 0.7076 | Mean MAE = 0.8886
[I 2026-06-08 16:39:51,720] Trial 0 finished with value: 0.7076346949089506 and parameters: {'C': 0.25137391690465716, 'epsilon': 0.043301188946018504, 'gamma': 'scale'}. Best is trial 0 with value: 0.7076346949089506.
Trial 1 | Mean R2 = 0.7193 | Mean MAE = 0.8113
[I 2026-06-08 16:41:19,002] Trial 1 finished with value: 0.719336818225616 and parameters: {'C': 523.2806087916287, 'epsilon': 0.0067717544196994614, 'gamma': 'scale'}. Best is trial 1 with value: 0.719336818225616.
Trial 2 | Mean R2 = 0.7688 | Mean MAE = 0.7421
[I 2026-06-08 16:42:33,441] Trial 2 finished with value: 0.7688427938738284 and parameters: {'C': 1.7064424581777824, 'epsilon': 0.014793259005548719, 'gamma': 'auto'}. Best is trial 2 with value: 0.7688427938738284.
Trial 3 | Mean R2 = 0.7738 | Mean MAE = 0.7244
[I 2026-06-08 16:44:55,163] Trial 3 finished with value: 0.7738474817112408 and parameters: {'C': 2.792937027763292, 'epsilon': 0.010969074683409297, 'gamma': '

In [8]:
print("Best Mean CV R2:")
print(study.best_value)

print("\nBest Parameters:")
print(study.best_params)

Best Mean CV R2:
0.7759133287355825

Best Parameters:
{'C': 4.200888546955157, 'epsilon': 0.06164595418860771, 'gamma': 'scale'}


In [9]:

best_svr_pipeline = Pipeline([

    (
        "preprocessing",
        preprocessor
    ),

    (
        "model",

        SVR(

            **study.best_params,

            kernel="rbf"
        )
    )

])

best_svr_pipeline.fit(
    X_train,
    y_train
)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('imputer', ...), ('variance_filter', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None


In [10]:
# TRAIN PREDICTIONS

y_train_pred = best_svr_pipeline.predict(
    X_train
)

# TEST PREDICTIONS

y_test_pred = best_svr_pipeline.predict(
    X_test
)

In [11]:


train_r2 = r2_score(
    y_train,
    y_train_pred
)

test_r2 = r2_score(
    y_test,
    y_test_pred
)

train_rmse = np.sqrt(
    mean_squared_error(
        y_train,
        y_train_pred
    )
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

train_mae = mean_absolute_error(
    y_train,
    y_train_pred
)

test_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

print("\n TUNED SVR ")

print("\nTrain R2 :", train_r2)
print("Train RMSE :", train_rmse)
print("Train MAE :", train_mae)

print("\nTest R2 :", test_r2)
print("Test RMSE :", test_rmse)
print("Test MAE :", test_mae)


 TUNED SVR 

Train R2 : 0.8980741717055652
Train RMSE : 0.7552635933996131
Train MAE : 0.3519028222589842

Test R2 : 0.7931043274212117
Test RMSE : 1.0810926209606628
Test MAE : 0.6998452963652577


In [12]:
print("Best Parameters:")
print(study.best_params)

Best Parameters:
{'C': 4.200888546955157, 'epsilon': 0.06164595418860771, 'gamma': 'scale'}


In [19]:
import pandas as pd

trials_df = study.trials_dataframe()

trials_df["Mean_CV_MAE"] = [
    t.user_attrs.get("mean_mae")
    for t in study.trials
]

trials_df["Mean_CV_R2"] = [
    t.user_attrs.get("mean_r2")
    for t in study.trials
]

final_trials_df = trials_df[
    [
        "number",
        "Mean_CV_R2",
        "Mean_CV_MAE",
        "params_C",
        "params_epsilon",
        "params_gamma",
        "state"
    ]
].copy()

final_trials_df.columns = [
    "Trial_Number",
    "Mean_CV_R2",
    "Mean_CV_MAE",
    "C",
    "Epsilon",
    "Gamma",
    "Status"
]

final_trials_df.to_csv(
    "svr_optuna_trials.csv",
    index=False
)

final_trials_df.head()

,Trial_Number,Mean_CV_R2,Mean_CV_MAE,C,Epsilon,Gamma,Status
0,0,0.707635,0.888614,0.251374,0.043301,scale,COMPLETE
1,1,0.719337,0.811343,523.280609,0.006772,scale,COMPLETE
2,2,0.768843,0.742066,1.706442,0.014793,auto,COMPLETE
3,3,0.773847,0.724407,2.792937,0.010969,auto,COMPLETE
4,4,0.773576,0.719298,8.420394,0.128377,scale,COMPLETE


In [20]:
final_trials_df.head(30)

,Trial_Number,Mean_CV_R2,Mean_CV_MAE,C,Epsilon,Gamma,Status
0,0,0.707635,0.888614,0.251374,0.043301,scale,COMPLETE
1,1,0.719337,0.811343,523.280609,0.006772,scale,COMPLETE
2,2,0.768843,0.742066,1.706442,0.014793,auto,COMPLETE
3,3,0.773847,0.724407,2.792937,0.010969,auto,COMPLETE
4,4,0.773576,0.719298,8.420394,0.128377,scale,COMPLETE
5,5,0.750597,0.762125,50.503214,0.004435,scale,COMPLETE
6,6,0.768180,0.762195,2.840805,0.558306,scale,COMPLETE
7,7,0.739660,0.820161,0.529713,0.207801,auto,COMPLETE
8,8,0.417666,1.372049,0.011944,0.102901,scale,COMPLETE
9,9,0.737105,0.806174,234.393556,0.504754,scale,COMPLETE


In [17]:
trials_df.shape

(30, 13)

In [21]:
joblib.dump(
    best_svr_pipeline,
    "best_svr_pipeline.pkl"
)

['best_svr_pipeline.pkl']

In [33]:
svr = pd.read_csv("svr_optuna_trials.csv")

svr.loc[
    svr["Mean_CV_R2"].idxmax()
]

Trial_Number          23
Mean_CV_R2      0.775913
Mean_CV_MAE     0.716478
C               4.200889
Epsilon         0.061646
Gamma              scale
Status          COMPLETE
Name: 23, dtype: object